# B2-023-generative-models-diffusion — Practice p12 — Solution

**Type:** constrained-coding · **Difficulty:** advanced · **Concepts:** stable-diffusion

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Supplied cell (copied verbatim from the statement)

In [ ]:
import math
import torch
from torch import nn

SEED = 20261015
U = torch.tensor([[0.2, -0.4], [1.0, 0.0]])
C = torch.tensor([[0.5, 0.1], [0.0, 1.0]])
LABELS5 = torch.tensor([0, 1, 2, 0, 1])
KEEP5 = torch.tensor([True, True, False, True, False])
QF = torch.tensor([[1.0, 2.0, 0.5], [0.0, -1.0, 0.3], [2.0, 0.0, -1.0]])
LABELS = torch.tensor([0, 1, 1])
KEEP = torch.tensor([True, True, False])


def set_literal_weights(block):
    with torch.no_grad():
        block.q_proj.weight.copy_(torch.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]]))
        block.q_proj.bias.copy_(torch.tensor([0.0, 0.5]))
        block.k_proj.weight.copy_(torch.eye(2))
        block.v_proj.weight.copy_(torch.tensor([[2.0, 0.0], [0.0, 1.0]]))
        block.class_tokens.weight.copy_(torch.tensor([[1.0, 0.0], [0.0, 1.0]]))
        block.null_token.copy_(torch.tensor([0.5, -0.5]))


ATOL = 1e-6
RTOL = 1e-6

## Solution

**Guidance.** `cfg_combine` is the one-line formula $\varepsilon_u+w(\varepsilon_c-\varepsilon_u)$ with a shape guard; $w=0$ returns $\varepsilon_u$ and $w=1$ returns $\varepsilon_c$ (up to rounding).

**Cross-attention block.** The latent's features own the single query; the two condition tokens (class token first, then the shared learned null token) own the keys and values (B2-021's modal-ownership rule).
For a dropped example (`keep == False`) the class-token score is replaced by $-\infty$ with `masked_fill`, so the softmax gives it weight exactly $0$ and the null token weight exactly $1$, and the output is exactly `v_proj(null_token)`.
One private method `_attend` runs the computation and returns every intermediate, so `forward`, `attention_weights`, and the shape ledger all observe the same computation.

In [ ]:
def cfg_combine(eps_uncond, eps_cond, w):
    if eps_uncond.shape != eps_cond.shape:
        raise ValueError("eps_uncond and eps_cond must have the same shape")
    return eps_uncond + w * (eps_cond - eps_uncond)


class CrossAttentionConditioner(nn.Module):
    def __init__(self, query_in, dim, num_classes):
        super().__init__()
        self.q_proj = nn.Linear(query_in, dim)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.class_tokens = nn.Embedding(num_classes, dim)
        self.null_token = nn.Parameter(torch.zeros(dim))
        self.dim = dim

    def _attend(self, query_feats, labels, keep, zero_score_bug=False):
        B, dim = query_feats.shape[0], self.dim
        tokens = torch.stack([self.class_tokens(labels), self.null_token.expand(B, dim)], dim=1)  # (B,2,dim)
        q = self.q_proj(query_feats).unsqueeze(1)                                                 # (B,1,dim)
        k = self.k_proj(tokens)                                                                   # (B,2,dim)
        v = self.v_proj(tokens)                                                                   # (B,2,dim)
        scores = q @ k.transpose(1, 2) / math.sqrt(dim)                                           # (B,1,2)
        drop_class = torch.stack([~keep, torch.zeros_like(keep)], dim=1).unsqueeze(1)            # (B,1,2)
        fill = 0.0 if zero_score_bug else float("-inf")
        weights = torch.softmax(scores.masked_fill(drop_class, fill), dim=-1)                     # (B,1,2)
        out = (weights @ v).squeeze(1)                                                            # (B,dim)
        return {"query": q, "tokens": tokens, "keys": k, "values": v,
                "scores": scores, "weights": weights, "output": out}

    def forward(self, query_feats, labels, keep):
        return self._attend(query_feats, labels, keep)["output"]

    def attention_weights(self, query_feats, labels, keep):
        return self._attend(query_feats, labels, keep)["weights"]


def attention_shape_ledger(block, query_feats, labels, keep):
    parts = block._attend(query_feats, labels, keep)
    names = ("query", "tokens", "keys", "values", "scores", "weights", "output")
    return [(name, tuple(parts[name].shape)) for name in names]

### Probes 1–2

In [ ]:
g0, g1, g15 = cfg_combine(U, C, 0.0), cfg_combine(U, C, 1.0), cfg_combine(U, C, 1.5)
flipped = lambda u, c, w: u - w * (c - u)

torch.manual_seed(SEED)
block = CrossAttentionConditioner(3, 16, 3)
QF5 = torch.randn(5, 3, generator=torch.Generator().manual_seed(SEED))
ledger = attention_shape_ledger(block, QF5, LABELS5, KEEP5)
print("w=1.5:", g15.tolist())
print("ledger:", ledger)

### Probe 3: literal block

Hand check of row 0 ($q_f=(1,2,0.5)$, label $0$, kept): $q=(1,\,2.5)$; the class token $(1,0)$ has key $(1,0)$ and value $(2,0)$; the null token $(0.5,-0.5)$ has key $(0.5,-0.5)$ and value $(1,-0.5)$.
Scores $=(1/\sqrt2,\ -0.75/\sqrt2)=(0.707107,-0.530330)$, weights $=(0.775118,0.224882)$, output $=0.775118(2,0)+0.224882(1,-0.5)=(1.775118,-0.112441)$.
Row 2 is dropped, so its output is the null value $(1,-0.5)$ exactly.

In [ ]:
lit = CrossAttentionConditioner(3, 2, 2)
set_literal_weights(lit)
lit_out = lit(QF, LABELS, KEEP)
with torch.no_grad():
    null_value = lit.v_proj(lit.null_token)
lit_zero_score = lit._attend(QF, LABELS, KEEP, zero_score_bug=True)["output"]
print("literal output:", lit_out.tolist())
print("zero-score dropped row:", lit_zero_score[2].tolist())

### Probes 4–6

In [ ]:
weights5 = block.attention_weights(QF5, LABELS5, KEEP5)
dropped = ~KEEP5

out5 = block(QF5, LABELS5, KEEP5)
labels_dropped_changed = LABELS5.clone()
labels_dropped_changed[dropped] = (labels_dropped_changed[dropped] + 1) % 3
out5_dropped_changed = block(QF5, labels_dropped_changed, KEEP5)
labels_kept_changed = LABELS5.clone()
labels_kept_changed[0] = 1                     # example 0 is kept; its label changes 0 -> 1
out5_kept_changed = block(QF5, labels_kept_changed, KEEP5)

block.zero_grad(set_to_none=True)
block(QF5, LABELS5, KEEP5).sum().backward()
class_grad = block.class_tokens.weight.grad
print("weights:", weights5.squeeze(1).tolist())
print("class-token grad row norms:", class_grad.norm(dim=1).tolist())

### Answer check

In [ ]:
# Probe 1
assert torch.equal(g0, U)
assert torch.allclose(g1, C, atol=1e-6, rtol=1e-6)
assert torch.allclose(g15, torch.tensor([[0.65, 0.35], [-0.5, 1.5]]), atol=1e-6, rtol=1e-6)
assert torch.equal(flipped(U, C, 0.0), g0)
assert not torch.allclose(flipped(U, C, 1.0), C, atol=1e-6, rtol=1e-6)
try:
    cfg_combine(U, C[:1], 1.0)
    raise AssertionError("shape mismatch must raise")
except ValueError:
    pass
# Probe 2
assert ledger == [("query", (5, 1, 16)), ("tokens", (5, 2, 16)), ("keys", (5, 2, 16)), ("values", (5, 2, 16)),
                  ("scores", (5, 1, 2)), ("weights", (5, 1, 2)), ("output", (5, 16))]
# Probe 3
assert torch.allclose(lit_out, torch.tensor([[1.7751175165, -0.1124412194], [0.6295601130, 0.0556598604], [1.0, -0.5]]),
                      atol=1e-6, rtol=1e-6)
assert torch.equal(lit_out[2], null_value)
# Probe 4
assert torch.allclose(weights5.sum(dim=-1), torch.ones(5, 1), atol=1e-6, rtol=1e-6)
assert all(torch.equal(weights5[i, 0], torch.tensor([0.0, 1.0])) for i in torch.nonzero(dropped).flatten().tolist())
# Probe 5
assert torch.equal(out5_dropped_changed[dropped], out5[dropped])
assert not torch.equal(out5_kept_changed[0], out5[0])
# Probe 6
assert torch.equal(class_grad[2], torch.zeros(16))
assert bool((class_grad[0] != 0).any()) and bool((class_grad[1] != 0).any())
# Probe 7
assert not torch.allclose(lit_zero_score[2], lit_out[2], atol=1e-6, rtol=1e-6)